<a href="https://colab.research.google.com/github/supmine/COMS4705_Project_Code/blob/main/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSM8K eval — COMS 4705

Runs one baseline from `eval/run_eval.py` on Colab and saves results to Google Drive.

1. Runtime → Change runtime type → **GPU** (T4 works; L4 is faster and supports bf16).
2. Run cells top to bottom. After the install cell, restart the session once (Runtime → Restart session) and continue from step 3.

| Baseline | Name | Needs `ADAPTER` |
|---|---|---|
| 1 | Zero-shot CoT | no |
| 2 | Frozen solver–reviser | no |
| 3 | Single-agent GRPO | yes |
| 4 | Single-agent GRPO + self-revision | yes (same adapter for both steps) |

## 1. GPU, Drive, code

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO = "/content/COMS4705_Project_Code"
if not os.path.exists(REPO):
    !git clone https://github.com/supmine/COMS4705_Project_Code.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1

## 2. Install (first time per session), then restart the session

In [ ]:
!pip install -q -r requirements.txt
# Then: Runtime -> Restart session, and continue from step 3.

## 3. Settings

Edit this cell. For baselines 3 and 4, put the LoRA adapter on Drive and set `ADAPTER` to its folder.
For trained adapters, set `SEED` to the training seed so `plot_results.py` can average seeds.

In [ ]:
import os, torch
%cd /content/COMS4705_Project_Code

BASELINE = 2          # 1, 2, 3 or 4
ADAPTER = None        # e.g. "/content/drive/MyDrive/coms4705/adapters/single_grpo_seed0"
SEED = 0
SPLIT = "test"        # "test" for reported numbers, "dev" for tuning
OUT_DIR = "/content/drive/MyDrive/coms4705/outputs"
FIG_DIR = "/content/drive/MyDrive/coms4705/figures"

# T4 (compute capability 7.5) has no bfloat16 -> load in float16.
DTYPE = "half" if torch.cuda.get_device_capability()[0] < 8 else "auto"
print(torch.cuda.get_device_name(), "-> dtype", DTYPE)

ARGS = f"--baseline {BASELINE} --seed {SEED} --dtype {DTYPE}"
if ADAPTER:
    ARGS += f" --adapter {ADAPTER}"
os.makedirs(OUT_DIR, exist_ok=True)
print("python -m eval.run_eval", ARGS)

## 4. Unit tests (gold solutions grade 100%, prompt matches Qwen's template)

In [ ]:
!python -m pytest -q tests/

## 5. Smoke test: 20 dev problems

Check the printed prompt: it must start with `<|im_start|>system` and Qwen's default system message.

In [ ]:
!python -m eval.run_eval {ARGS} --split dev --limit 20 --out_dir /content/smoke
!python -m eval.metrics /content/smoke/*/

## 6. Full run (writes `OUT_DIR/<run_id>/predictions.jsonl` + `config.json`)

In [ ]:
!python -m eval.run_eval {ARGS} --split {SPLIT} --out_dir {OUT_DIR}

## 7. Metrics and figures over every run in `OUT_DIR`

Includes teammates' runs if they save to the same shared Drive folder.

In [ ]:
!python -m eval.metrics {OUT_DIR}/*/

In [ ]:
!python eval/plot_results.py {OUT_DIR}/*/ --output_dir {FIG_DIR}
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv(f"{FIG_DIR}/summary.csv"))
for name in ("acc_vs_tokens", "flip_rates"):
    if os.path.exists(f"{FIG_DIR}/{name}.png"):
        display(Image(f"{FIG_DIR}/{name}.png"))

## 8. Read 20 random problems by eye before sharing numbers

In [ ]:
import glob, json, random
run_dir = sorted(glob.glob(f"{OUT_DIR}/b{BASELINE}_{SPLIT}*"))[-1]
rows = [json.loads(l) for l in open(f"{run_dir}/predictions.jsonl")]
for r in random.Random(0).sample(rows, 20):
    print(f"=== qid {r['qid']}  gold {r['gold']}  final_correct {r['final_correct']}")
    for t in r["turns"]:
        print(f"--- {t['agent']}: pred={t['pred']} method={t['method']} finish={t['finish_reason']}")
        print(t["text"][-400:])